# Chapter 3: Recompute the region evaluator

Chapter 3 defines the evaluator: membership in a region, observed Brier loss and the contrast between the region and the rest. This notebook checks that the saved confirmation summaries of the fresh run follow from its observations. Use the full fresh confirmation sample to verify Brier loss inside and outside every frozen region. Region membership was selected before these outcomes were drawn.

**Evidence.** Run 20260923T220909Z repeated the regional study end to end on two machines. `spark-fe12` ran numerical discovery with three new optimizer seeds (74101 to 74103), one live structural proposal and a fresh 200,000-observation confirmation sample. `spark-ef84` ran 70 live answer tests and nine report drafts with `claude-sonnet-5`. The population and frozen logistic predictor are those of the chapter companions; the seeds, proposal and confirmation sample are new, so these numbers should not be combined with the historical run's.

## Setup and source checks

The setup cell locates the data root, the directory whose relative layout the evidence records use. It uses `MODEL_VALIDATION_ROOT` when that variable is set. Otherwise it searches for a repository checkout containing the current directory and then for the copy installed with the `aiv` package. `R` is this run's folder under the data root, and `read` loads a JSON file from it.

`reference.json` was exported from the run's regional results and records their SHA-256 digest. The final assertion recomputes that digest, so every value used below comes from the unchanged results of this run. This notebook reads recorded outputs; rerunning it makes no language-model call.

In [1]:
from pathlib import Path
import json, hashlib, os, platform
import numpy as np
import pandas as pd

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
LIVE = ROOT / "book/experiments/regional_live"
R = ROOT / "book/experiments/regional_live/runs/20260923T220909Z"
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
print("Run:", R.relative_to(ROOT))
read=lambda p: json.loads((R/p).read_text())
results=read('regional/results.json')
reference=read('reference.json')
seeds=reference['seeds']
assert hashlib.sha256((R/'regional/results.json').read_bytes()).hexdigest()==reference['source_sha256']


Python: 3.12.3
Data root found from: repository checkout
Run: book/experiments/regional_live/runs/20260923T220909Z


## Recompute every confirmation summary

`confirmation.npz` holds the confirmation outcomes `y`, the frozen predictor's probabilities `p` and one Boolean membership mask per frozen region. The cell computes the Brier loss $L_i=(y_i-p_i)^2$ of every observation and, for each region, asserts that the mask's mean equals the saved share, that the mean loss of members equals the saved Brier loss and that the difference between member and non-member means equals the saved contrast.

In [2]:
a=np.load(R/'regional/confirmation.npz')
loss=(a['y']-a['p'])**2
for key,row in results['confirmation'].items():
 mask=a[key]
 assert np.isclose(mask.mean(),row['share'])
 assert np.isclose(loss[mask].mean(),row['brier'])
 assert np.isclose(loss[mask].mean()-loss[~mask].mean(),row['contrast'])
print('All',len(results['confirmation']),'regions verified on',len(loss),'observations')

All 13 regions verified on 200000 observations


**Read the output.** All thirteen summaries are reproduced on the 200,000 confirmation observations. The saved results therefore follow directly from the observations and the frozen masks. Because each mask was computed from features before outcomes were drawn, none of these measurements influenced which region was selected.